In [8]:
"""
A script that takes a the output layer of a model as input and prunes it according to the given reduction args
"""

from harmonic_inference.data.data_types import TRIAD_REDUCTION
from harmonic_inference.utils.harmonic_utils import get_chord_label_list
from harmonic_inference.data.data_types import PitchType
import torch
from torch.nn.utils import prune
from harmonic_inference.models.chord_classifier_models import SimpleChordClassifier


target_labels = get_chord_label_list(pitch_type=PitchType.TPC, use_inversions=False, reduction=TRIAD_REDUCTION)

full_labels = get_chord_label_list(pitch_type=PitchType.TPC, use_inversions=True, reduction=None)
# iterate over full labels, if full_label == target_label -> 1, else 0 (one-hots)

# num_classes = len(target_labels)



def create_pruning_mask(target_labels: list, full_labels: list) -> torch.Tensor:
    """
    creates one-hot version of full_labels where 1 keeps class and 0 prunes class
    """
    mask = [1 if label in target_labels else 0 for label in full_labels]

    return torch.tensor(mask, dtype=torch.float32) # bool?



def prune_output_layer(layer:torch.nn.Linear, mask:torch.tensor):
    """
    prunes output classes by zeroing out corresponding weights and biases
    """

    # expand 1D mask to 2D for the weight matrix
    weight_mask = mask.unsqueeze(1).expand_as(layer.weight)


    prune.custom_from_mask(layer, name="weight", mask=weight_mask)
    if layer.bias is not None:
        prune.custom_from_mask(layer, name="bias", mask=mask)


if __name__ == "__main__":

    model = SimpleChordClassifier.load_from_checkpoint(checkpoint_path=r"checkpoints-beat-best/ccm/lightning_logs/version_0/checkpoints/epoch=310-step=2413359.ckpt")


    # print("Non-zero weights:", torch.count_nonzero(model.fc2.weight).item())

    # Apply pruning

    mask = create_pruning_mask(target_labels, full_labels)

    prune_output_layer(model.fc2, mask)

    print("Success! Output layer sucessfully pruned")


    # print("Non-zero weights:", torch.count_nonzero(model.fc2.weight).item())

    torch.save(model, "checkpoints/ccm/lightning_logs/pruned_ccm_model.pt")

# print("Mask shape after:", model.fc2.weight_mask.shape)
# print("Active weights after:", torch.sum(model.fc2.weight_mask).item())
# print("Pruned weights after:", (model.fc2.weight_mask == 0).sum().item())


Lightning automatically upgraded your loaded checkpoint from v1.5.5 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file checkpoints-beat-best/ccm/lightning_logs/version_0/checkpoints/epoch=310-step=2413359.ckpt`


Success! Output layer sucessfully pruned


## everything below is not relevant

In [ ]:
"""
A script that takes a the output (softmax) layer of a model as input and prunes it according to the given reduction args
"""

from harmonic_inference.data.data_types import TRIAD_REDUCTION
from harmonic_inference.utils.harmonic_utils import get_chord_label_list
from harmonic_inference.data.data_types import PitchType
import torch
from pathlib import Path
from torch.nn.utils import prune

target_labels = get_chord_label_list(pitch_type=PitchType.TPC, use_inversions=False, reduction=TRIAD_REDUCTION)

full_labels = get_chord_label_list(pitch_type=PitchType.TPC, use_inversions=True, reduction=None)
# iterate over full labels, if full_label == target_label -> 1, else 0 (one-hots)
num_classes = len(target_labels)

# load model

def apply_pruning(
        dir_model: str | Path
        ):
    
    dir_model = Path(dir_model)
    model = torch.nn.LSTM()
    checkpoint = torch.load(dir_model, map_location="cpu")

def reduce_to_target_label(label: str, target_labels: list):
    """
    reduce full labels like: F##+M7 -> F##+
    by stripping extensions from full_labels
    """

    suffixes = [
        "64", "65", "43", "2", "6", 
        "M7", "M65", "M43", "M2",
        "m7", "m65", "m43", "m2",
        "7", "%7", "+7", "%", "+",
        "%65", "%43", "%2"
    ]

    if label in target_labels:
        return label
    
    for suffix in sorted(suffixes, key=len, reverse=True):
        if label.endswith(suffix):
            reduced_label = label[: -len(suffix)]
            if reduced_label in target_labels:
                return reduced_label

    if label in target_labels:
        return label
    raise ValueError(f"Couldn't reduce label {label} to a target label")


def build_target_mapping(full_labels, target_labels) -> list:
    """
    Return a list of the new (=reduced) target labels
    """
    reduced = []

    for full_label in full_labels:
        reduced_label = reduce_to_target_label(full_label, target_labels)
        reduced.append(reduced_label)
    return reduced

mapping = {}
for label in full_labels:
    reduced = reduce_to_target_label(label, target_labels)
    mapping[label] = reduced


print(mapping, sep="\s")

def reduce_softmax(model, full_labels, target_labels, layer_name="output"):
    """
    reduce final output layer from full labels (len=approx. 1400) 
    to target labels (len= approx 140)
    """

    old_layer = getattr(model, layer_name)

    if not isinstance(old_layer, torch.nn.Linear):
        raise TypeError(f"{layer_name} is not a linear layer")


In [ ]:
"""
A script that takes a the output (softmax) layer of a model as input and prunes it according to the given reduction args
"""

from harmonic_inference.data.data_types import TRIAD_REDUCTION
from harmonic_inference.utils.harmonic_utils import get_chord_label_list
from harmonic_inference.data.data_types import PitchType
import torch
from pathlib import Path
from torch.nn.utils import prune
from collections import Counter
target_labels = get_chord_label_list(pitch_type=PitchType.TPC, use_inversions=False, reduction=TRIAD_REDUCTION)

full_labels = get_chord_label_list(pitch_type=PitchType.TPC, use_inversions=True, reduction=None)
# iterate over full labels, if full_label == target_label -> 1, else 0 (one-hots)
num_classes = len(target_labels)


one_hot_labels = []

for label in full_labels:
    if label in target_labels:
        one_hot_labels.append(1)
    else:
        one_hot_labels.append(0)

one_hot_labels = torch.tensor(one_hot_labels)

print(one_hot_labels)

def create_pruning_mask(target_labels, full_labels):
    """
    creates one-hot version of full_labels where 1 keeps class and 0 prunes class
    """

    mask = [1 if label in target_labels else 0 for label in full_labels]
    return torch.tensor(mask,dtype=torch.bool)

def prune_output_layer(layer:torch.nn.Linear, mask:torch.tensor):
    """
    prunes output classes by zeroing out corresponding weights and biases
    """
    with torch.no_grad():
        layer.weight[~mask] = 0.0
        if layer.bias is not None:
            layer.bias[~mask] = 0.0

